# Aula 02 — Resposta Tarefa - Titanic: Pipeline “mínimo profissional” (reprodutível)

**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  

## 🎯 Objetivo
Treinar um modelo de ML no Titanic de forma **reprodutível** usando:
- **Dados → Treino → Teste**
- **Pipeline + ColumnTransformer** (tratando missing + categorias)
- **Baseline:** Logistic Regression
- **Melhoria simples:** Random Forest
- **Avaliação:** accuracy, classification_report, matriz de confusão + TN/FP/FN/TP  
- **Interpretação humana** dos erros (FP/FN)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
# 1) Carregar o dataset Titanic e remover a coluna redundante "alive"
df_t = sns.load_dataset("titanic").drop(columns=["alive"])

display(df_t.head())
print("Shape:", df_t.shape)

print("\nTipos das colunas:")
print(df_t.dtypes)

print("\nFaltantes por coluna (top 10):")
display(df_t.isna().sum().sort_values(ascending=False).head(10))

print("\nDistribuição do target (survived):")
display(df_t["survived"].value_counts())
display(df_t["survived"].value_counts(normalize=True).rename("proporção"))

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,True


Shape: (891, 14)

Tipos das colunas:
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alone              bool
dtype: object

Faltantes por coluna (top 10):


,0
deck,688
age,177
embarked,2
embark_town,2
pclass,0
survived,0
parch,0
sibsp,0
sex,0
fare,0



Distribuição do target (survived):


,count
survived,
0,549
1,342


,proporção
survived,
0,0.616162
1,0.383838


## 📚 Dicionário de dados (resumo)
- **Target (y):** `survived` (0 = não sobreviveu, 1 = sobreviveu)
- Exemplos de features:
  - Numéricas: `age`, `fare`, `sibsp`, `parch`, `pclass`
  - Categóricas: `sex`, `embarked`, `class`, `who`, `deck`, `embark_town`, `alone`, `adult_male`

⚠️ Observação: há **missing values** (ex.: `age`, `deck`, `embarked`), e há colunas categóricas/texto.
Por isso, precisamos de **Pipeline + ColumnTransformer**.

In [ ]:
# 2) Separar X e y
y = df_t["survived"]
X = df_t.drop(columns=["survived"])

print("X shape:", X.shape)


X shape: (891, 13)


In [ ]:
print("y shape:", y.shape)


y shape: (891,)


In [ ]:
display(X.head())

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,False
1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,False
2,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,True
3,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,False
4,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,True


In [ ]:
# 3) Identificar colunas por tipo
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

print("Numéricas:", list(num_cols))
print("Categóricas:", list(cat_cols))

Numéricas: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categóricas: ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


In [ ]:
# 4) Dividir em treino e teste (reprodutível e com estratificação do target)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train.shape, "Teste:", X_test.shape)


Treino: (712, 13) Teste: (179, 13)


In [ ]:
print("Proporção do target (treino):")
display(y_train.value_counts(normalize=True).rename("proporção"))


Proporção do target (treino):


,proporção
survived,
0,0.616573
1,0.383427


In [ ]:
print("Proporção do target (teste):")
display(y_test.value_counts(normalize=True).rename("proporção"))

Proporção do target (teste):


,proporção
survived,
0,0.614525
1,0.385475


## 🛠️ Por que precisamos de Pipeline + ColumnTransformer?

Em dados reais (Titanic), temos:
- **Missing values** (NaN): precisamos imputar (preencher de forma consistente)
- **Colunas categóricas** (texto/bool): precisamos converter para números (One-Hot)
- **Reprodutibilidade**: tudo deve acontecer **dentro do Pipeline**
  - O pré-processamento “aprende” no treino
  - E “aplica” no teste
  - Evita *data leakage*

In [ ]:
# Pipelines por tipo de dado
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# O "Chef Principal": ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

preprocess

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alone'],
      dtype='object'))])

In [ ]:
def avaliar_classificacao(nome, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)  # por padrão: [[TN, FP],[FN, TP]] para labels 0/1
    tn, fp, fn, tp = cm.ravel()

    print(f"\n=== {nome} ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred))
    print("Matriz de confusão [[TN, FP],[FN, TP]]:")
    print(cm)
    print(f"\nTN={tn} | FP={fp} | FN={fn} | TP={tp}")

    # Interpretação humana (Titanic)
    print("\nInterpretação humana (classe 1 = sobreviveu):")
    print(f"- FN={fn}: pessoas que SOBREVIVERAM, mas o modelo previu 'não sobreviveu' (perdeu sobreviventes).")
    print(f"- FP={fp}: pessoas que NÃO sobreviveram, mas o modelo previu 'sobreviveu' (falso alarme de sobrevivência).")

    return {
        "acc": acc,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "cm": cm
    }

## 🚀 Baseline — Logistic Regression (com Pipeline completo)

Estrutura do “sanduíche”:
- **prep**: preprocess (imputer + scaler + onehot)
- **clf**: LogisticRegression

Depois:
1) fit no treino  
2) predict no teste  
3) avaliar (accuracy + report + matriz + TN/FP/FN/TP)

In [ ]:
model_baseline_lr = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

print("Treinando baseline (LogisticRegression)...")
model_baseline_lr.fit(X_train, y_train)

pred_baseline_lr = model_baseline_lr.predict(X_test)

res_lr = avaliar_classificacao("Baseline — LogisticRegression", y_test, pred_baseline_lr)
acc_baseline_lr = res_lr["acc"]
tn_baseline_lr, fp_baseline_lr, fn_baseline_lr, tp_baseline_lr = res_lr["tn"], res_lr["fp"], res_lr["fn"], res_lr["tp"]

Treinando baseline (LogisticRegression)...

=== Baseline — LogisticRegression ===
Accuracy: 0.8268

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       110
           1       0.79      0.75      0.77        69

    accuracy                           0.83       179
   macro avg       0.82      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179

Matriz de confusão [[TN, FP],[FN, TP]]:
[[96 14]
 [17 52]]

TN=96 | FP=14 | FN=17 | TP=52

Interpretação humana (classe 1 = sobreviveu):
- FN=17: pessoas que SOBREVIVERAM, mas o modelo previu 'não sobreviveu' (perdeu sobreviventes).
- FP=14: pessoas que NÃO sobreviveram, mas o modelo previu 'sobreviveu' (falso alarme de sobrevivência).


## 🧐 Interpretação (objetiva) — Baseline

1) Acurácia diz “quantos acertos no total”, mas **não mostra** qual erro é mais comum.
2) Para Titanic, um erro bem importante é o **FN**:
   - o modelo diz “não sobreviveu” quando, na verdade, **sobreviveu**.
3) A matriz de confusão e o classification_report mostram esse trade-off.

## 🔧 Melhoria simples (1 mudança)
Trocar apenas o classificador:
- de **LogisticRegression**
- para **RandomForestClassifier**

Importante:
- O pré-processamento continua o mesmo (`preprocess`)
- Isso mantém o experimento **justo**: só mudou o modelo

In [ ]:
model_improved_rf = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE))
])

print("Treinando melhoria (RandomForestClassifier)...")
model_improved_rf.fit(X_train, y_train)

pred_improved_rf = model_improved_rf.predict(X_test)

res_rf = avaliar_classificacao("Melhoria — RandomForestClassifier", y_test, pred_improved_rf)
acc_improved_rf = res_rf["acc"]
tn_improved_rf, fp_improved_rf, fn_improved_rf, tp_improved_rf = res_rf["tn"], res_rf["fp"], res_rf["fn"], res_rf["tp"]

Treinando melhoria (RandomForestClassifier)...

=== Melhoria — RandomForestClassifier ===
Accuracy: 0.8212

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       110
           1       0.82      0.68      0.75        69

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.82       179

Matriz de confusão [[TN, FP],[FN, TP]]:
[[100  10]
 [ 22  47]]

TN=100 | FP=10 | FN=22 | TP=47

Interpretação humana (classe 1 = sobreviveu):
- FN=22: pessoas que SOBREVIVERAM, mas o modelo previu 'não sobreviveu' (perdeu sobreviventes).
- FP=10: pessoas que NÃO sobreviveram, mas o modelo previu 'sobreviveu' (falso alarme de sobrevivência).


In [ ]:
print("=== Comparação Final ===")
print(f"Acurácia (LogisticRegression):      {acc_baseline_lr:.4f}")
print(f"Acurácia (RandomForestClassifier):  {acc_improved_rf:.4f}")
print("-" * 70)
print(f"TN - LR: {tn_baseline_lr} | RF: {tn_improved_rf}")
print(f"FP - LR: {fp_baseline_lr} | RF: {fp_improved_rf}")
print(f"FN - LR: {fn_baseline_lr} | RF: {fn_improved_rf}")
print(f"TP - LR: {tp_baseline_lr} | RF: {tp_improved_rf}")
print("-" * 70)

print("Leitura rápida (classe 1 = sobreviveu):")
print("- FN menor = melhor para 'não perder sobreviventes'.")
print("- FP menor = menos 'alarmes falsos' de sobrevivência.")

=== Comparação Final ===
Acurácia (LogisticRegression):      0.8268
Acurácia (RandomForestClassifier):  0.8212
----------------------------------------------------------------------
TN - LR: 96 | RF: 100
FP - LR: 14 | RF: 10
FN - LR: 17 | RF: 22
TP - LR: 52 | RF: 47
----------------------------------------------------------------------
Leitura rápida (classe 1 = sobreviveu):
- FN menor = melhor para 'não perder sobreviventes'.
- FP menor = menos 'alarmes falsos' de sobrevivência.


## ✅ Respostas (interpretação em linguagem humana)

### 1) O modelo erra mais em qual classe?
Geralmente, no Titanic, muitos modelos erram mais a **classe 1 (sobreviveu)**, porque:
- o dataset é “difícil” e tem fatores históricos/ocultos
- sobreviventes podem ter padrões menos óbvios que “não sobreviventes”

A evidência está em:
- **recall da classe 1** no `classification_report`
- e no valor de **FN** na matriz de confusão

---

### 2) O que aconteceu com FP quando você tentou reduzir FN (trade-off)?
Quando um modelo tenta “pegar mais sobreviventes” (reduzir FN), é comum:
- **aumentar FP** (chutar “sobreviveu” mais vezes)

Isso é o trade-off clássico:
- reduzir FN pode custar mais FP
- e reduzir FP pode custar mais FN

A comparação objetiva está nos valores de **FP e FN** entre baseline e melhoria.

---

### 3) Uma limitação do dataset
Limitações comuns do Titanic:
- **Missing values** (ex.: `deck`, `age`, `embarked`)
- **viés histórico** (ex.: “mulheres e crianças primeiro”, classe social)
- **variáveis ausentes** importantes (ex.: proximidade do bote, treino para nadar, localização exata, tempo de reação)

# README — Correação (Titanic)

> Adicionar aspas



- **Baseline:** LogisticRegression (modelo simples, bom ponto de partida e interpretável).
- **Missing values:** numéricas com mediana; categóricas com moda (SimpleImputer no Pipeline).
- **Melhoria:** RandomForestClassifier (captura relações não-lineares e interações).
- **Comparação FP/FN:** verificar a matriz de confusão e comparar FP e FN entre os modelos (trade-off).
- **Limitação do dataset:** missing values + vieses históricos + falta de variáveis relevantes (contexto real do acidente).